# RQ4 — IA × Qualidade e Reprodutibilidade em Notebooks Jupyter

**Pergunta de pesquisa**: *Em que medida o uso de ferramentas de Inteligência Artificial na geração de código influencia a qualidade e a reprodutibilidade dos notebooks?*

Este notebook usa os CSVs gerados por `collect_notebooks.py` e `execute_notebooks.py` para:
1. Carregar e integrar dados de coleta e execução;
2. Construir variáveis *proxy* para **uso de IA**;
3. Definir e calcular métricas de **qualidade** e **reprodutibilidade**;
4. Conduzir análises estatísticas (descritas, inferência, regressões logísticas, *robustness checks*);
5. Reportar efeitos (odds ratios) e intervalos de confiança.

## 0. Parâmetros e caminhos dos dados
Ajuste os caminhos abaixo para apontar aos CSVs gerados pelos seus scripts.

In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

COLLECTION_CSV = 'data/outputs/collection.csv'  # CSV do collect_notebooks.py
EXECUTION_CSV  = 'data/outputs/execution_results.csv'  # CSV do execute_notebooks.py
assert os.path.exists(COLLECTION_CSV), f'Arquivo não encontrado: {COLLECTION_CSV}'
assert os.path.exists(EXECUTION_CSV), f'Arquivo não encontrado: {EXECUTION_CSV}'
# === Configuração de caminhos ===


print('OK: arquivos encontrados')

AssertionError: Arquivo não encontrado: data/outputs/collection.csv

## 1. Carregar dados
Integração `collection` × `execution` por `(repo_full_name, file_path)`.

In [ ]:
def load_csv(path: str) -> pd.DataFrame:
    return pd.read_csv(path)

coll = load_csv(COLLECTION_CSV)
exe  = load_csv(EXECUTION_CSV)

# chaves de junção
keys = ['repo_full_name','file_path']
merged = pd.merge(coll, exe, on=keys, how='inner', suffixes=('_coll','_exec'))
print(merged.shape)
merged.head(3)

## 2. *Proxy* de uso de IA (exposição)

### 2.1 Múltiplas medidas de exposição à IA

A detecção de código gerado por IA é um desafio metodológico não trivial (Wang et al., 2024). Aqui adotamos uma abordagem **conservadora e transparente**: construímos múltiplos proxies heurísticos baseados em padrões observáveis, cada um capturando diferentes aspectos de possível uso de IA:

**Proxies individuais**:
1. **`ai_marker_found`**: Menções explícitas no Markdown (e.g., "ChatGPT", "Claude", "Copilot")
2. **`triple_backticks_in_code`**: Artefatos de cópia-cola (```python em células de código)
3. **`ai_comment_density`**: Proporção de comentários/código (IA tende a gerar comentários verbosos)
4. **`code_style_uniformity`**: Consistência estilística (IA tende a formatar uniformemente)

**Medidas agregadas**:
- **`ai_pattern_score`**: Escore contínuo (0–1) ponderando múltiplos sinais
- **`ai_exposure_binary`**: Classificação binária conservadora (qualquer sinal positivo)
- **`ai_exposure_level`**: Classificação ordinal (0=none, 1=weak, 2=moderate, 3=strong)

### 2.2 Limitações da mensuração

⚠️ **Validade de construto**: Proxies heurísticos têm limitações conhecidas:
- **Falsos positivos**: Menções de IA em contexto educacional; código bem documentado por humanos
- **Falsos negativos**: Código gerado por IA sem menção explícita; desenvolvedores que removem artefatos
- **Viés de seleção**: Apenas sinais superficiais; não detecta assistência sutil (autocompletar)

**Trabalhos futuros** devem integrar classificadores validados baseados em ML (Wang et al., 2024: "An Empirical Study on Automatically Detecting AI-Generated Source Code") que analisam padrões sintáticos, semânticos e estilísticos mais profundos.


In [ ]:
# === Funções auxiliares ===
def to_bool(x):
    if pd.isna(x):
        return False
    s = str(x).strip().lower()
    return s in ('true','1','yes','y','t')

def as_float(x):
    try:
        return float(x)
    except (ValueError, TypeError):
        return np.nan

# === Proxies individuais ===
# 1. Marcadores explícitos de IA (da coleta)
merged['ai_marker_found'] = merged['ai_marker_found'].apply(to_bool).astype(int)

# 2. Artefatos de cópia-cola (da coleta)
merged['triple_backticks_in_code'] = merged['triple_backticks_in_code'].apply(to_bool).astype(int)

# 3. Densidade de comentários (proxy: alto valor sugere documentação estilo IA)
# Usamos a proporção de células com outputs vs. células de código como proxy simplificado
# (notebooks IA tendem a ter mais células executadas com documentação)
merged['n_code_safe'] = merged['n_code'].apply(as_float).fillna(1).clip(lower=1)
merged['n_markdown_safe'] = merged['n_markdown'].apply(as_float).fillna(0)
merged['ai_comment_density'] = (merged['n_markdown_safe'] / merged['n_code_safe']).clip(upper=5.0)
# Normalizar para 0-1 (threshold: >2 markdown/code é alto)
merged['ai_comment_density_norm'] = (merged['ai_comment_density'] / 2.0).clip(upper=1.0)

# 4. Uniformidade de estilo (proxy: notebooks com ordem executação perfeita são mais uniformes)
# IA tende a gerar notebooks com execução limpa (sem pulos, ordem clara)
merged['has_unambiguous_order_tmp'] = merged['has_unambiguous_order'].apply(to_bool).astype(int)
merged['out_of_order_tmp'] = merged['out_of_order'].apply(to_bool).astype(int)
merged['ai_style_uniformity'] = (
    merged['has_unambiguous_order_tmp'] * 0.6 +
    (1 - merged['out_of_order_tmp']) * 0.4
)

# === Medidas agregadas ===
# Pattern score: média ponderada de sinais (0–1)
# Pesos baseados em confiança: marcadores explícitos > artefatos > sinais indiretos
merged['ai_pattern_score'] = (
    merged['ai_marker_found'] * 0.40 +
    merged['triple_backticks_in_code'] * 0.30 +
    merged['ai_comment_density_norm'] * 0.15 +
    merged['ai_style_uniformity'] * 0.15
).clip(lower=0, upper=1)

# Binary exposure (conservador: qualquer sinal forte)
merged['ai_exposure_binary'] = (
    (merged['ai_marker_found'] == 1) | 
    (merged['triple_backticks_in_code'] == 1)
).astype(int)

# Ordinal exposure level (0=none, 1=weak, 2=moderate, 3=strong)
def compute_ai_level(row):
    score = row['ai_pattern_score']
    if row['ai_marker_found'] == 1 and row['triple_backticks_in_code'] == 1:
        return 3  # strong: múltiplos sinais diretos
    elif row['ai_marker_found'] == 1 or row['triple_backticks_in_code'] == 1:
        return 2  # moderate: pelo menos um sinal direto
    elif score >= 0.3:
        return 1  # weak: apenas sinais indiretos acima do threshold
    else:
        return 0  # none

merged['ai_exposure_level'] = merged.apply(compute_ai_level, axis=1)

# Para compatibilidade com código existente, mantemos 'ai_exposure' como alias de binary
merged['ai_exposure'] = merged['ai_exposure_binary']

print("=== Distribuição das medidas de exposição à IA ===")
print("\nBinary exposure:")
print(merged['ai_exposure_binary'].value_counts().sort_index())
print("\nOrdinal exposure level:")
print(merged['ai_exposure_level'].value_counts().sort_index())
print(f"\nPattern score (contínuo): mean={merged['ai_pattern_score'].mean():.3f}, "
      f"std={merged['ai_pattern_score'].std():.3f}, "
      f"median={merged['ai_pattern_score'].median():.3f}")

In [ ]:
# Visualização das distribuições de exposição à IA
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 1. Pattern score distribution
axes[0, 0].hist(merged['ai_pattern_score'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('AI Pattern Score')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of AI Pattern Score (continuous)')
axes[0, 0].axvline(merged['ai_pattern_score'].mean(), color='red', linestyle='--', label=f'Mean: {merged["ai_pattern_score"].mean():.3f}')
axes[0, 0].legend()

# 2. Ordinal level
level_counts = merged['ai_exposure_level'].value_counts().sort_index()
axes[0, 1].bar(level_counts.index, level_counts.values, edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('AI Exposure Level')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('AI Exposure Level (ordinal: 0=none, 1=weak, 2=moderate, 3=strong)')
axes[0, 1].set_xticks([0, 1, 2, 3])

# 3. Individual proxies comparison
proxy_means = pd.DataFrame({
    'Marker Found': [merged['ai_marker_found'].mean()],
    'Triple Backticks': [merged['triple_backticks_in_code'].mean()],
    'Comment Density': [merged['ai_comment_density_norm'].mean()],
    'Style Uniformity': [merged['ai_style_uniformity'].mean()]
}).T
proxy_means.columns = ['Proportion']
proxy_means.plot(kind='barh', ax=axes[1, 0], legend=False, color='steelblue', edgecolor='black')
axes[1, 0].set_xlabel('Mean Value (0-1)')
axes[1, 0].set_title('Mean Values of Individual AI Proxies')
axes[1, 0].set_xlim(0, 1)

# 4. Correlation between proxies
proxy_cols = ['ai_marker_found', 'triple_backticks_in_code', 'ai_comment_density_norm', 'ai_style_uniformity']
corr = merged[proxy_cols].corr()
im = axes[1, 1].imshow(corr, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
axes[1, 1].set_xticks(range(len(proxy_cols)))
axes[1, 1].set_yticks(range(len(proxy_cols)))
axes[1, 1].set_xticklabels(['Marker', 'Backticks', 'Comment', 'Style'], rotation=45, ha='right')
axes[1, 1].set_yticklabels(['Marker', 'Backticks', 'Comment', 'Style'])
axes[1, 1].set_title('Correlation between AI Proxies')
# Add correlation values
for i in range(len(proxy_cols)):
    for j in range(len(proxy_cols)):
        text = axes[1, 1].text(j, i, f'{corr.iloc[i, j]:.2f}',
                              ha="center", va="center", color="black", fontsize=8)
plt.colorbar(im, ax=axes[1, 1])

plt.tight_layout()
plt.show()

print("\n=== Estatísticas descritivas dos proxies individuais ===")
print(merged[['ai_marker_found', 'triple_backticks_in_code', 'ai_comment_density_norm', 'ai_style_uniformity']].describe())


## 3. Métricas de **qualidade** (proxies)

### 3.1 Fundamentação teórica das dimensões de qualidade

A qualidade de notebooks Jupyter é multidimensional (Pimentel et al., 2019; Rule et al., 2019). Operacionalizamos **seis dimensões** com base na literatura de reprodutibilidade computacional:

**D1. Completude de execução** (30%)
- **Métrica**: `percent_code_executed` (% de células de código com execution_count definido)
- **Justificativa**: Notebooks executados demonstram funcionalidade verificável (Pimentel et al., 2019: "A Large-Scale Study About Quality and Reproducibility of Jupyter Notebooks")
- **Peso**: Maior peso, pois é o indicador mais direto de notebook "funcionando"

**D2. Clareza estrutural** (20%)
- **Métrica**: `has_unambiguous_order` (execução sequencial sem ambiguidades)
- **Justificativa**: Ordem de execução clara é crítica para reprodutibilidade (Pimentel 2019; Rule 2019: "Ten Simple Rules for Reproducible Research in Jupyter Notebooks")
- **Peso**: Alto, reflete design intencional vs. exploração caótica

**D3. Ausência de erros** (20%)
- **Métrica**: `!outputs_error` (nenhuma célula com output do tipo error)
- **Justificativa**: Outputs limpos indicam código robusto; erros salvos sugerem incomplete workflow
- **Peso**: Alto, critério objetivo de qualidade mínima

**D4. Práticas de engenharia** (15%)
- **Métrica**: `uses_testing_module` (importa unittest, pytest, etc.)
- **Justificativa**: Testes automatizados são padrão-ouro em engenharia de software (Wilson et al., 2014: "Best Practices for Scientific Computing")
- **Peso**: Moderado, diferencia projetos profissionais de scripts exploratórios

**D5. Documentação de dependências** (10%)
- **Métrica**: `deps_any` (presença de requirements.txt, setup.py ou Pipfile)
- **Justificativa**: Gestão explícita de dependências é requisito básico de reprodutibilidade (Rule 2019, regra 5)
- **Peso**: Moderado, mas menos crítico que execução propriamente dita

**D6. Portabilidade** (5%)
- **Métrica**: `!has_abs_data_path` (ausência de caminhos absolutos no código)
- **Justificativa**: Caminhos absolutos impedem execução em ambientes diferentes
- **Peso**: Menor, pois pode ser corrigido facilmente

### 3.2 Penalizações por fragmentação
- **Métrica**: `n_skips_total` (lacunas na sequência de execution_count)
- **Penalização**: -10% (normalizado), indicador de desenvolvimento não-linear ou células deletadas

### 3.3 Análise de componentes
Reportamos cada dimensão **separadamente** antes da agregação, permitindo:
- Comparações dimensão-a-dimensão entre grupos (IA vs. não-IA)
- Identificação de trade-offs (e.g., maior completude mas menor clareza)
- Validação de colinearidade via PCA/análise fatorial

**Composite score** (0–1): soma ponderada das dimensões, com interpretação transparente.

In [ ]:
# === Normalização de variáveis brutas ===
for col in ['percent_code_executed','n_skips_total','n_cells_with_output']:
    if col not in merged.columns or col in ['n_code_safe', 'n_markdown_safe']:
        merged[col] = 0
    merged[col] = merged[col].apply(as_float)

for col in ['has_unambiguous_order','outputs_error','uses_testing_module','deps_any','has_abs_data_path']:
    if col not in merged.columns:
        merged[col] = False
    merged[col] = merged[col].apply(to_bool).astype(int)

# === Construção das dimensões de qualidade (0-1 cada) ===
# D1: Completude de execução (30%)
merged['quality_d1_execution'] = (merged['percent_code_executed'].fillna(0) / 100.0).clip(0, 1)

# D2: Clareza estrutural (20%)
merged['quality_d2_structure'] = merged['has_unambiguous_order'].astype(float)

# D3: Ausência de erros (20%)
merged['quality_d3_no_errors'] = (1 - merged['outputs_error']).astype(float)

# D4: Práticas de engenharia (15%)
merged['quality_d4_testing'] = merged['uses_testing_module'].astype(float)

# D5: Documentação de dependências (10%)
merged['quality_d5_deps'] = merged['deps_any'].astype(float)

# D6: Portabilidade (5%)
merged['quality_d6_portable'] = (1 - merged['has_abs_data_path']).astype(float)

# Penalização por fragmentação (normalizada 0-1, onde 1 = sem fragmentação)
merged['quality_penalty_skips'] = (1 - (merged['n_skips_total'].fillna(0).clip(0, 10) / 10.0)).clip(0, 1)

# === Composite quality score (teoricamente justificado) ===
# Pesos: D1=0.30, D2=0.20, D3=0.20, D4=0.15, D5=0.10, D6=0.05
# Penalização de skips: -0.10 do total
merged['quality_score'] = (
    merged['quality_d1_execution'] * 0.30 +
    merged['quality_d2_structure'] * 0.20 +
    merged['quality_d3_no_errors'] * 0.20 +
    merged['quality_d4_testing'] * 0.15 +
    merged['quality_d5_deps'] * 0.10 +
    merged['quality_d6_portable'] * 0.05 +
    merged['quality_penalty_skips'] * 0.10
).clip(0, 1)

print("=== Estatísticas descritivas do Quality Score ===")
print(merged['quality_score'].describe())
print("\nDistribuição por quartis:")
print(merged['quality_score'].quantile([0, 0.25, 0.5, 0.75, 1.0]))

# === Análise de componentes individuais ===
print("\n=== Componentes de qualidade (médias) ===")
component_means = pd.DataFrame({
    'D1_Execution (30%)': merged['quality_d1_execution'].mean(),
    'D2_Structure (20%)': merged['quality_d2_structure'].mean(),
    'D3_NoErrors (20%)': merged['quality_d3_no_errors'].mean(),
    'D4_Testing (15%)': merged['quality_d4_testing'].mean(),
    'D5_Deps (10%)': merged['quality_d5_deps'].mean(),
    'D6_Portable (5%)': merged['quality_d6_portable'].mean(),
    'Penalty_Skips': merged['quality_penalty_skips'].mean()
}, index=['Mean']).T
print(component_means)

# === Correlação entre componentes (verificar colinearidade) ===
quality_components = [
    'quality_d1_execution', 'quality_d2_structure', 'quality_d3_no_errors',
    'quality_d4_testing', 'quality_d5_deps', 'quality_d6_portable'
]
print("\n=== Matriz de correlação entre componentes ===")
corr_components = merged[quality_components].corr()
print(corr_components.round(3))

In [ ]:
# === Visualização das dimensões de qualidade ===
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribuição do composite score
axes[0, 0].hist(merged['quality_score'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].set_xlabel('Quality Score (0-1)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Composite Quality Score')
axes[0, 0].axvline(merged['quality_score'].mean(), color='red', linestyle='--', 
                    label=f'Mean: {merged["quality_score"].mean():.3f}')
axes[0, 0].axvline(merged['quality_score'].median(), color='green', linestyle='--',
                    label=f'Median: {merged["quality_score"].median():.3f}')
axes[0, 0].legend()

# 2. Contribuição de cada dimensão (médias)
dim_contributions = pd.DataFrame({
    'D1: Execution': merged['quality_d1_execution'].mean() * 0.30,
    'D2: Structure': merged['quality_d2_structure'].mean() * 0.20,
    'D3: No Errors': merged['quality_d3_no_errors'].mean() * 0.20,
    'D4: Testing': merged['quality_d4_testing'].mean() * 0.15,
    'D5: Deps': merged['quality_d5_deps'].mean() * 0.10,
    'D6: Portable': merged['quality_d6_portable'].mean() * 0.05,
}, index=['Contribution']).T
dim_contributions.plot(kind='barh', ax=axes[0, 1], legend=False, color='coral', edgecolor='black')
axes[0, 1].set_xlabel('Weighted Contribution to Quality Score')
axes[0, 1].set_title('Mean Weighted Contribution by Dimension')
axes[0, 1].set_xlim(0, 0.30)

# 3. Mapa de calor da correlação entre componentes
quality_comp_cols = ['quality_d1_execution', 'quality_d2_structure', 'quality_d3_no_errors',
                     'quality_d4_testing', 'quality_d5_deps', 'quality_d6_portable']
corr_qual = merged[quality_comp_cols].corr()
im = axes[1, 0].imshow(corr_qual, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
axes[1, 0].set_xticks(range(len(quality_comp_cols)))
axes[1, 0].set_yticks(range(len(quality_comp_cols)))
axes[1, 0].set_xticklabels(['D1', 'D2', 'D3', 'D4', 'D5', 'D6'], rotation=0)
axes[1, 0].set_yticklabels(['D1', 'D2', 'D3', 'D4', 'D5', 'D6'])
axes[1, 0].set_title('Correlation Matrix of Quality Dimensions')
for i in range(len(quality_comp_cols)):
    for j in range(len(quality_comp_cols)):
        axes[1, 0].text(j, i, f'{corr_qual.iloc[i, j]:.2f}',
                       ha="center", va="center", color="black", fontsize=9)
plt.colorbar(im, ax=axes[1, 0])

# 4. PCA: Scree plot (validação de dimensionalidade)
X_quality = merged[quality_comp_cols].dropna()
if len(X_quality) > 10:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_quality)
    pca = PCA()
    pca.fit(X_scaled)
    explained_var = pca.explained_variance_ratio_
    cumulative_var = np.cumsum(explained_var)
    
    axes[1, 1].bar(range(1, len(explained_var)+1), explained_var, alpha=0.7, 
                   label='Individual', color='skyblue', edgecolor='black')
    axes[1, 1].plot(range(1, len(cumulative_var)+1), cumulative_var, 'ro-', 
                    label='Cumulative', linewidth=2)
    axes[1, 1].set_xlabel('Principal Component')
    axes[1, 1].set_ylabel('Explained Variance Ratio')
    axes[1, 1].set_title('PCA Scree Plot (Quality Dimensions)')
    axes[1, 1].legend()
    axes[1, 1].set_xticks(range(1, len(explained_var)+1))
    axes[1, 1].grid(axis='y', alpha=0.3)
    
    print("\n=== PCA: Variância explicada por componente ===")
    for i, var in enumerate(explained_var, 1):
        print(f"PC{i}: {var:.3f} ({cumulative_var[i-1]:.3f} cumulative)")
    print(f"\nPrimeiros 2 componentes explicam {cumulative_var[1]:.1%} da variância")
    print(f"Primeiros 3 componentes explicam {cumulative_var[2]:.1%} da variância")
else:
    axes[1, 1].text(0.5, 0.5, 'Insufficient data for PCA', 
                   ha='center', va='center', transform=axes[1, 1].transAxes)

plt.tight_layout()
plt.show()


## 4. Métricas de **reprodutibilidade**
Baseado no executor:
- `exec_ok` (execução completa sem erro — *headless*, timeout real);
- `outputs_equal` quando o notebook original foi salvo (`original_found=True`): hash de outputs do original igual ao do executado (proxy objetiva de reprodutibilidade).

In [ ]:
for col in ['exec_ok','original_found','outputs_equal']:
    if col in merged.columns:
        merged[col] = merged[col].apply(to_bool).astype(int)

merged[['exec_ok','original_found','outputs_equal']].mean().to_frame('mean').T

## 5. Análise descritiva

### 5.1 Comparação de grupos (IA vs. não-IA)

Comparamos distribuições de **qualidade** e **reprodutibilidade** entre notebooks com e sem exposição à IA, reportando:
- **Médias e desvios-padrão** por grupo
- **Effect sizes** (Cohen's d) para quantificar magnitude das diferenças
- **Testes de significância** (t-test ou Mann-Whitney U se não-normal)
- **Visualizações** de distribuições (violin plots, KDE)

**Interpretação de effect size** (Cohen, 1988):
- |d| < 0.2: negligível
- 0.2 ≤ |d| < 0.5: pequeno
- 0.5 ≤ |d| < 0.8: médio
- |d| ≥ 0.8: grande

In [ ]:
from scipy import stats

# === Função para calcular Cohen's d ===
def cohens_d(group1, group2):
    """Calcula effect size (Cohen's d) entre dois grupos."""
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(), group2.var()
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    return (group1.mean() - group2.mean()) / pooled_std if pooled_std > 0 else 0

# === Comparação descritiva por grupo ===
metrics_to_compare = [
    'quality_score', 'quality_d1_execution', 'quality_d2_structure', 'quality_d3_no_errors',
    'quality_d4_testing', 'quality_d5_deps', 'exec_ok', 'outputs_equal'
]

# Separar grupos
g_no_ai = merged[merged['ai_exposure_binary'] == 0]
g_ai = merged[merged['ai_exposure_binary'] == 1]

print("=== Tamanhos dos grupos ===")
print(f"Sem IA: n={len(g_no_ai)} | Com IA: n={len(g_ai)}")
print(f"Proporção com IA: {len(g_ai) / len(merged):.1%}\n")

# Tabela de comparação
comparison_results = []
for metric in metrics_to_compare:
    if metric not in merged.columns:
        continue
    
    vals_no_ai = g_no_ai[metric].dropna()
    vals_ai = g_ai[metric].dropna()
    
    if len(vals_no_ai) < 2 or len(vals_ai) < 2:
        continue
    
    mean_no_ai = vals_no_ai.mean()
    mean_ai = vals_ai.mean()
    std_no_ai = vals_no_ai.std()
    std_ai = vals_ai.std()
    
    # Cohen's d
    d = cohens_d(vals_ai, vals_no_ai)
    
    # T-test (two-sided)
    t_stat, p_val = stats.ttest_ind(vals_ai, vals_no_ai, equal_var=False)
    
    # Mann-Whitney U (non-parametric alternative)
    u_stat, p_val_mw = stats.mannwhitneyu(vals_ai, vals_no_ai, alternative='two-sided')
    
    comparison_results.append({
        'Metric': metric,
        'Mean_NoAI': mean_no_ai,
        'SD_NoAI': std_no_ai,
        'Mean_AI': mean_ai,
        'SD_AI': std_ai,
        'Diff': mean_ai - mean_no_ai,
        'Cohen_d': d,
        'p_ttest': p_val,
        'p_mannwhitney': p_val_mw
    })

comparison_df = pd.DataFrame(comparison_results)
comparison_df['Effect_size_interp'] = comparison_df['Cohen_d'].apply(
    lambda d: 'large' if abs(d) >= 0.8 else ('medium' if abs(d) >= 0.5 else ('small' if abs(d) >= 0.2 else 'negligible'))
)

print("=== Comparação descritiva: IA vs. Não-IA ===")
print(comparison_df[['Metric', 'Mean_NoAI', 'Mean_AI', 'Diff', 'Cohen_d', 'Effect_size_interp', 'p_ttest']].round(4))
print("\nLegenda: Cohen_d > 0 indica média maior no grupo IA; < 0 indica média maior no grupo Não-IA")
print("p_ttest: p-value do teste t (H0: médias iguais); significância convencional: p < 0.05")

In [ ]:
# === Visualizações de distribuições: IA vs. Não-IA ===
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Prepara dados para violin plot
merged_viz = merged.copy()
merged_viz['AI_Group'] = merged_viz['ai_exposure_binary'].map({0: 'No AI', 1: 'AI'})

# Métricas chave para visualizar
viz_metrics = [
    ('quality_score', 'Composite Quality Score'),
    ('quality_d1_execution', 'D1: Execution Completeness'),
    ('quality_d3_no_errors', 'D3: No Errors'),
    ('exec_ok', 'Successful Execution'),
    ('outputs_equal', 'Output Reproducibility'),
    ('ai_pattern_score', 'AI Pattern Score')
]

for idx, (metric, title) in enumerate(viz_metrics):
    ax = axes[idx // 3, idx % 3]
    
    if metric not in merged_viz.columns:
        ax.text(0.5, 0.5, f'{metric} not available', ha='center', va='center')
        ax.set_title(title)
        continue
    
    # Violin plot
    data_to_plot = [
        merged_viz[merged_viz['AI_Group'] == 'No AI'][metric].dropna(),
        merged_viz[merged_viz['AI_Group'] == 'AI'][metric].dropna()
    ]
    
    parts = ax.violinplot(data_to_plot, positions=[0, 1], showmeans=True, showmedians=True)
    
    # Colorir violins
    for pc in parts['bodies']:
        pc.set_facecolor('lightblue')
        pc.set_alpha(0.7)
    
    # Adicionar pontos de média
    means = [d.mean() for d in data_to_plot]
    ax.scatter([0, 1], means, color='red', s=100, zorder=3, label='Mean')
    
    # Configurar eixos
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['No AI', 'AI'])
    ax.set_ylabel('Value')
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.3)
    
    # Adicionar effect size no título
    if len(data_to_plot[0]) > 1 and len(data_to_plot[1]) > 1:
        d = cohens_d(data_to_plot[1], data_to_plot[0])
        ax.text(0.5, 0.95, f"Cohen's d = {d:.3f}", transform=ax.transAxes, 
                ha='center', va='top', fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

# === Correlação entre covariáveis (verificar confundimento) ===
print("\n=== Matriz de correlação: covariáveis e desfechos ===")
corr_vars = ['ai_exposure_binary', 'quality_score', 'exec_ok', 'repo_stars', 'n_code', 'percent_code_executed']
corr_vars_available = [v for v in corr_vars if v in merged.columns]

if len(corr_vars_available) > 2:
    corr_matrix = merged[corr_vars_available].corr()
    
    plt.figure(figsize=(10, 8))
    im = plt.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
    plt.colorbar(im)
    plt.xticks(range(len(corr_vars_available)), corr_vars_available, rotation=45, ha='right')
    plt.yticks(range(len(corr_vars_available)), corr_vars_available)
    plt.title('Correlation Matrix: Covariates and Outcomes')
    
    # Adicionar valores
    for i in range(len(corr_vars_available)):
        for j in range(len(corr_vars_available)):
            plt.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                    ha="center", va="center", color="black", fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    print(corr_matrix.round(3))
else:
    print("Insufficient variables for correlation matrix")

## 6. Inferência: regressão logística (efeito de IA)

### 6.1 Especificação do modelo

Estimamos modelos logísticos para dois desfechos binários:

**Modelo 1: Execução bem-sucedida**
\[
\text{logit}(P(\text{exec\_ok} = 1)) = \beta_0 + \beta_1 \cdot \text{ai\_exposure} + \sum \beta_j \cdot \text{covariate}_j
\]

**Modelo 2: Reprodutibilidade de outputs**
\[
\text{logit}(P(\text{outputs\_equal} = 1 \mid \text{original\_found})) = \beta_0 + \beta_1 \cdot \text{ai\_exposure} + \sum \beta_j \cdot \text{covariate}_j
\]

**Covariáveis de controle** (potenciais confundidores):
- `repo_stars`: popularidade/maturidade do repositório
- `n_code`: tamanho do notebook (complexidade)
- `deps_any`: presença de documentação de dependências
- `percent_code_executed`: completude de execução prévia (medida estrutural)

**Parâmetro de interesse**: \(\beta_1\) (efeito de exposição à IA)

### 6.2 Diagnósticos reportados

- **Odds Ratios (OR) e IC95%**: interpretação multiplicativa do efeito
- **Pseudo-R²** (McFadden, Nagelkerke): ajuste do modelo
- **VIF (Variance Inflation Factor)**: multicolinearidade (VIF > 5 indica problema)
- **ROC curve e AUC**: capacidade preditiva
- **Hosmer-Lemeshow test**: calibração do modelo

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import roc_curve, auc

# === Função estendida com diagnósticos ===
def logit_with_diagnostics(df: pd.DataFrame, y_col: str, x_cols: list, model_name: str):
    """
    Ajusta modelo logístico com diagnósticos completos:
    - VIF para multicolinearidade
    - Pseudo-R² (McFadden e Nagelkerke)
    - ROC/AUC
    - Hosmer-Lemeshow test (simplificado)
    """
    # Preparar dados
    d = df.dropna(subset=[y_col]+x_cols).copy()
    if len(d) < 10:
        print(f"[{model_name}] Insufficient data: n={len(d)}")
        return None, None, None
    
    y = d[y_col].astype(int)
    X = d[x_cols].copy()
    
    # 1. VIF (antes de adicionar constante)
    print(f"\n{'='*60}")
    print(f"MODELO: {model_name}")
    print(f"{'='*60}")
    print(f"Sample size: n={len(d)} | Events: {y.sum()} ({y.mean():.1%})")
    
    print("\n--- Variance Inflation Factors (VIF) ---")
    vif_data = pd.DataFrame()
    vif_data["Variable"] = x_cols
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(x_cols))]
    print(vif_data)
    print("Nota: VIF > 5 indica multicolinearidade problemática; VIF > 10 é severa")
    
    # 2. Ajustar modelo
    X_with_const = sm.add_constant(X, has_constant='add')
    model = sm.Logit(y, X_with_const).fit(disp=False, maxiter=100)
    
    # 3. Odds Ratios e ICs
    params = model.params
    conf = model.conf_int()
    or_df = np.exp(pd.DataFrame({
        'OR': params,
        'CI_low': conf[0],
        'CI_high': conf[1],
        'p_value': model.pvalues
    }))
    
    print("\n--- Odds Ratios e Intervalos de Confiança (95%) ---")
    print(or_df.round(4))
    print("\nInterpretação OR:")
    print("  OR > 1: aumento nas chances do desfecho (efeito positivo)")
    print("  OR < 1: redução nas chances do desfecho (efeito negativo)")
    print("  OR = 1: sem efeito")
    
    # 4. Pseudo-R²
    print("\n--- Model Fit ---")
    llnull = model.llnull
    llf = model.llf
    mcfadden_r2 = 1 - (llf / llnull)
    
    # Nagelkerke R² (Cox-Snell ajustado)
    n = len(y)
    cox_snell = 1 - np.exp((llnull - llf) * (2/n))
    max_cox_snell = 1 - np.exp(llnull * (2/n))
    nagelkerke_r2 = cox_snell / max_cox_snell if max_cox_snell > 0 else 0
    
    print(f"McFadden Pseudo-R²: {mcfadden_r2:.4f}")
    print(f"Nagelkerke Pseudo-R²: {nagelkerke_r2:.4f}")
    print(f"AIC: {model.aic:.2f} | BIC: {model.bic:.2f}")
    print(f"Log-Likelihood: {llf:.2f} (Null: {llnull:.2f})")
    
    # 5. ROC e AUC
    y_pred_proba = model.predict(X_with_const)
    if len(np.unique(y)) > 1:
        fpr, tpr, thresholds = roc_curve(y, y_pred_proba)
        roc_auc = auc(fpr, tpr)
        print(f"\nROC AUC: {roc_auc:.4f}")
        print("Interpretação AUC: 0.5=random, 0.7=fair, 0.8=good, 0.9=excellent")
    else:
        fpr, tpr, roc_auc = None, None, None
        print("\nROC AUC: N/A (apenas uma classe no desfecho)")
    
    # 6. Hosmer-Lemeshow (simplificado: decis de probabilidade)
    print("\n--- Calibration Check (Hosmer-Lemeshow-like) ---")
    d['_pred_prob'] = y_pred_proba
    d['_decile'] = pd.qcut(d['_pred_prob'], q=10, labels=False, duplicates='drop')
    hl_table = d.groupby('_decile').agg({
        y_col: ['sum', 'count', 'mean'],
        '_pred_prob': 'mean'
    })
    hl_table.columns = ['Observed', 'Total', 'Obs_Rate', 'Pred_Prob']
    print(hl_table.round(3))
    print("Calibração ideal: Obs_Rate ≈ Pred_Prob em cada decil")
    
    diagnostics = {
        'vif': vif_data,
        'mcfadden_r2': mcfadden_r2,
        'nagelkerke_r2': nagelkerke_r2,
        'roc_auc': roc_auc,
        'fpr': fpr,
        'tpr': tpr,
        'y_pred_proba': y_pred_proba,
        'y_true': y.values
    }
    
    return model, or_df, diagnostics

# === Preparar covariáveis ===
covars = ['ai_exposure_binary','repo_stars','n_code','deps_any','percent_code_executed']
for c in covars:
    if c not in merged.columns:
        merged[c] = 0
    # Normalizar repo_stars e n_code para evitar problemas numéricos
    if c == 'repo_stars':
        merged[c] = merged[c].apply(as_float).fillna(0)
        # Log-transform (stars + 1) para reduzir skew
        merged['repo_stars_log'] = np.log1p(merged[c])
    elif c == 'n_code':
        merged[c] = merged[c].apply(as_float).fillna(0).clip(lower=1)
        merged['n_code_log'] = np.log(merged[c])

# Usar versões log-transformadas para melhor ajuste
covars_final = ['ai_exposure_binary', 'repo_stars_log', 'n_code_log', 'deps_any', 'percent_code_executed']

# === Modelo 1: exec_ok ===
m_exec, or_exec, diag_exec = logit_with_diagnostics(
    merged, 'exec_ok', covars_final, "Modelo 1: Execução bem-sucedida (exec_ok)"
)

# === Modelo 2: outputs_equal (condicional a original_found) ===
merged_with_orig = merged[merged['original_found'] == 1].copy()
m_equal, or_equal, diag_equal = logit_with_diagnostics(
    merged_with_orig, 'outputs_equal', covars_final, "Modelo 2: Reprodutibilidade (outputs_equal)"
)

In [ ]:
# === Visualização: ROC Curves ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve para Modelo 1 (exec_ok)
if diag_exec and diag_exec['fpr'] is not None:
    axes[0].plot(diag_exec['fpr'], diag_exec['tpr'], color='darkorange', lw=2,
                 label=f"ROC curve (AUC = {diag_exec['roc_auc']:.3f})")
    axes[0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random classifier')
    axes[0].set_xlim([0.0, 1.0])
    axes[0].set_ylim([0.0, 1.05])
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title('ROC Curve: Modelo 1 (exec_ok)')
    axes[0].legend(loc="lower right")
    axes[0].grid(alpha=0.3)
else:
    axes[0].text(0.5, 0.5, 'ROC not available', ha='center', va='center')
    axes[0].set_title('ROC Curve: Modelo 1 (exec_ok)')

# ROC Curve para Modelo 2 (outputs_equal)
if diag_equal and diag_equal['fpr'] is not None:
    axes[1].plot(diag_equal['fpr'], diag_equal['tpr'], color='green', lw=2,
                 label=f"ROC curve (AUC = {diag_equal['roc_auc']:.3f})")
    axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random classifier')
    axes[1].set_xlim([0.0, 1.0])
    axes[1].set_ylim([0.0, 1.05])
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('ROC Curve: Modelo 2 (outputs_equal)')
    axes[1].legend(loc="lower right")
    axes[1].grid(alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'ROC not available', ha='center', va='center')
    axes[1].set_title('ROC Curve: Modelo 2 (outputs_equal)')

plt.tight_layout()
plt.show()


## 7. Análises de robustez e sensibilidade

### 7.1 Justificativa

Análises de robustez avaliam se os resultados principais são sensíveis a escolhas analíticas alternativas (Wohlin et al., 2012). Testamos:

**A. Especificação do modelo**
- A1: Remover `percent_code_executed` (potencial mediador da relação IA → qualidade → exec_ok)
- A2: Modelos alternativos (probit, complementary log-log)

**B. Definição da exposição**
- B1: Threshold sensitivity (usar `ai_exposure_level` ordinal com diferentes cortes)
- B2: Apenas marcadores diretos (`ai_marker_found` ou `triple_backticks_in_code`)

**C. Composição da amostra**
- C1: Balanceamento por downsampling (equalizar tamanhos dos grupos IA vs. não-IA)
- C2: Estratificação por características do repositório (stars quartiles)
- C3: Exclusão de outliers de qualidade (top/bottom 5%)

**D. Inferência**
- D1: Bootstrap confidence intervals (1000 replications)
- D2: Placebo test (permutação aleatória de `ai_exposure`)

### 7.2 Expectativas
- Efeitos devem ser qualitativamente similares (direção e significância)
- Placebo test deve produzir efeito nulo (validação interna)
- Magnitude pode variar, mas conclusões substantivas devem se manter

In [ ]:
rng = np.random.default_rng(42)

# Helper: função simplificada para ajustar modelo e retornar apenas OR do AI exposure
def quick_logit_or(df, y_col, x_cols, exposure_var='ai_exposure_binary'):
    """Ajusta logit e retorna OR e IC95% apenas para a variável de exposição."""
    d = df.dropna(subset=[y_col]+x_cols).copy()
    if len(d) < 10:
        return None, None, None, None
    y = d[y_col].astype(int)
    X = sm.add_constant(d[x_cols], has_constant='add')
    try:
        model = sm.Logit(y, X).fit(disp=False, maxiter=100)
        if exposure_var in model.params.index:
            or_val = np.exp(model.params[exposure_var])
            ci_low = np.exp(model.conf_int().loc[exposure_var, 0])
            ci_high = np.exp(model.conf_int().loc[exposure_var, 1])
            p_val = model.pvalues[exposure_var]
            return or_val, ci_low, ci_high, p_val
        else:
            return None, None, None, None
    except Exception as e:
        print(f"Model fit failed: {e}")
        return None, None, None, None

# Armazenar resultados de robustez
robustness_results = []

print("="*70)
print("ANÁLISES DE ROBUSTEZ: Efeito de AI_exposure sobre exec_ok")
print("="*70)

# --- A1: Sem percent_code_executed (mediador) ---
print("\n[A1] Sem percent_code_executed (potencial mediador)")
covars_no_pct = ['ai_exposure_binary', 'repo_stars_log', 'n_code_log', 'deps_any']
or_a1, ci_low_a1, ci_high_a1, p_a1 = quick_logit_or(merged, 'exec_ok', covars_no_pct)
if or_a1:
    print(f"  OR = {or_a1:.3f}, 95% CI [{ci_low_a1:.3f}, {ci_high_a1:.3f}], p = {p_a1:.4f}")
    robustness_results.append({'Check': 'A1_NoMediator', 'OR': or_a1, 'CI_low': ci_low_a1, 'CI_high': ci_high_a1, 'p': p_a1})

# --- A2: Modelo alternativo (Probit) ---
print("\n[A2] Modelo alternativo: Probit (em vez de Logit)")
d_probit = merged.dropna(subset=['exec_ok']+covars_final).copy()
if len(d_probit) >= 10:
    y_probit = d_probit['exec_ok'].astype(int)
    X_probit = sm.add_constant(d_probit[covars_final], has_constant='add')
    try:
        probit_model = sm.Probit(y_probit, X_probit).fit(disp=False, maxiter=100)
        # Probit coef não é diretamente OR, mas podemos reportar coef e p-value
        coef_probit = probit_model.params['ai_exposure_binary']
        p_probit = probit_model.pvalues['ai_exposure_binary']
        print(f"  Probit coef = {coef_probit:.3f}, p = {p_probit:.4f}")
        print("  (Nota: coef Probit > 0 indica efeito positivo; não é OR diretamente)")
        robustness_results.append({'Check': 'A2_Probit', 'OR': coef_probit, 'CI_low': np.nan, 'CI_high': np.nan, 'p': p_probit})
    except Exception:
        print("  Probit fit failed")

# --- B1: Threshold sensitivity (exposure level ordinal) ---
print("\n[B1] Threshold sensitivity: AI exposure level >= 2 (moderate+strong)")
merged['ai_exposure_mod_strong'] = (merged['ai_exposure_level'] >= 2).astype(int)
covars_b1 = ['ai_exposure_mod_strong', 'repo_stars_log', 'n_code_log', 'deps_any', 'percent_code_executed']
or_b1, ci_low_b1, ci_high_b1, p_b1 = quick_logit_or(merged, 'exec_ok', covars_b1, exposure_var='ai_exposure_mod_strong')
if or_b1:
    print(f"  OR = {or_b1:.3f}, 95% CI [{ci_low_b1:.3f}, {ci_high_b1:.3f}], p = {p_b1:.4f}")
    robustness_results.append({'Check': 'B1_ThresholdModerate', 'OR': or_b1, 'CI_low': ci_low_b1, 'CI_high': ci_high_b1, 'p': p_b1})

# --- B2: Apenas marcadores diretos ---
print("\n[B2] Apenas marcadores diretos (ai_marker_found)")
covars_b2 = ['ai_marker_found', 'repo_stars_log', 'n_code_log', 'deps_any', 'percent_code_executed']
or_b2, ci_low_b2, ci_high_b2, p_b2 = quick_logit_or(merged, 'exec_ok', covars_b2, exposure_var='ai_marker_found')
if or_b2:
    print(f"  OR = {or_b2:.3f}, 95% CI [{ci_low_b2:.3f}, {ci_high_b2:.3f}], p = {p_b2:.4f}")
    robustness_results.append({'Check': 'B2_DirectMarkersOnly', 'OR': or_b2, 'CI_low': ci_low_b2, 'CI_high': ci_high_b2, 'p': p_b2})

# --- C1: Balanceamento (downsampling) ---
print("\n[C1] Balanceamento por downsampling (equal group sizes)")
g0_bal = merged[merged['ai_exposure_binary']==0]
g1_bal = merged[merged['ai_exposure_binary']==1]
k_bal = min(len(g0_bal), len(g1_bal))
if k_bal >= 10:
    bal_sample = pd.concat([
        g0_bal.sample(k_bal, random_state=123, replace=False),
        g1_bal.sample(k_bal, random_state=123, replace=False)
    ])
    or_c1, ci_low_c1, ci_high_c1, p_c1 = quick_logit_or(bal_sample, 'exec_ok', covars_final)
    if or_c1:
        print(f"  OR = {or_c1:.3f}, 95% CI [{ci_low_c1:.3f}, {ci_high_c1:.3f}], p = {p_c1:.4f}")
        robustness_results.append({'Check': 'C1_Balanced', 'OR': or_c1, 'CI_low': ci_low_c1, 'CI_high': ci_high_c1, 'p': p_c1})
else:
    print("  Insufficient data for balanced sample")

# --- C2: Estratificação por repo_stars (high vs. low stars) ---
print("\n[C2] Estratificação por repo_stars (top 50% vs bottom 50%)")
median_stars = merged['repo_stars'].median()
high_stars = merged[merged['repo_stars'] >= median_stars]
low_stars = merged[merged['repo_stars'] < median_stars]
or_c2_high, ci_low_c2_high, ci_high_c2_high, p_c2_high = quick_logit_or(high_stars, 'exec_ok', covars_final)
or_c2_low, ci_low_c2_low, ci_high_c2_low, p_c2_low = quick_logit_or(low_stars, 'exec_ok', covars_final)
if or_c2_high:
    print(f"  High stars: OR = {or_c2_high:.3f}, 95% CI [{ci_low_c2_high:.3f}, {ci_high_c2_high:.3f}], p = {p_c2_high:.4f}")
    robustness_results.append({'Check': 'C2_HighStars', 'OR': or_c2_high, 'CI_low': ci_low_c2_high, 'CI_high': ci_high_c2_high, 'p': p_c2_high})
if or_c2_low:
    print(f"  Low stars: OR = {or_c2_low:.3f}, 95% CI [{ci_low_c2_low:.3f}, {ci_high_c2_low:.3f}], p = {p_c2_low:.4f}")
    robustness_results.append({'Check': 'C2_LowStars', 'OR': or_c2_low, 'CI_low': ci_low_c2_low, 'CI_high': ci_high_c2_low, 'p': p_c2_low})

# --- C3: Exclusão de outliers de qualidade (top/bottom 5%) ---
print("\n[C3] Exclusão de outliers de qualidade (remover top e bottom 5%)")
q05 = merged['quality_score'].quantile(0.05)
q95 = merged['quality_score'].quantile(0.95)
merged_trimmed = merged[(merged['quality_score'] >= q05) & (merged['quality_score'] <= q95)]
or_c3, ci_low_c3, ci_high_c3, p_c3 = quick_logit_or(merged_trimmed, 'exec_ok', covars_final)
if or_c3:
    print(f"  OR = {or_c3:.3f}, 95% CI [{ci_low_c3:.3f}, {ci_high_c3:.3f}], p = {p_c3:.4f}")
    robustness_results.append({'Check': 'C3_NoOutliers', 'OR': or_c3, 'CI_low': ci_low_c3, 'CI_high': ci_high_c3, 'p': p_c3})

# --- D1: Bootstrap CI (simplified: bootstrap OR estimates) ---
print("\n[D1] Bootstrap confidence intervals (n=200 iterations, simplified)")
# Simplificado: apenas 200 iterações para não consumir muito tempo
n_boot = 200
boot_ors = []
for i in range(n_boot):
    boot_sample = merged.sample(n=len(merged), replace=True, random_state=42+i)
    or_boot, _, _, _ = quick_logit_or(boot_sample, 'exec_ok', covars_final)
    if or_boot:
        boot_ors.append(or_boot)
if len(boot_ors) > 10:
    boot_ci_low = np.percentile(boot_ors, 2.5)
    boot_ci_high = np.percentile(boot_ors, 97.5)
    boot_mean = np.mean(boot_ors)
    print(f"  Bootstrap mean OR = {boot_mean:.3f}, 95% CI [{boot_ci_low:.3f}, {boot_ci_high:.3f}]")
    robustness_results.append({'Check': 'D1_Bootstrap', 'OR': boot_mean, 'CI_low': boot_ci_low, 'CI_high': boot_ci_high, 'p': np.nan})
else:
    print("  Bootstrap failed (insufficient successful fits)")

# --- D2: Placebo test (permutation) ---
print("\n[D2] Placebo test: permutação aleatória de ai_exposure")
perm_sample = merged.copy()
perm_sample['ai_exposure_binary'] = rng.permutation(perm_sample['ai_exposure_binary'].values)
or_d2, ci_low_d2, ci_high_d2, p_d2 = quick_logit_or(perm_sample, 'exec_ok', covars_final)
if or_d2:
    print(f"  OR = {or_d2:.3f}, 95% CI [{ci_low_d2:.3f}, {ci_high_d2:.3f}], p = {p_d2:.4f}")
    print("  EXPECTATIVA: OR ≈ 1.0 e p > 0.05 (efeito nulo quando exposição é aleatória)")
    robustness_results.append({'Check': 'D2_Placebo', 'OR': or_d2, 'CI_low': ci_low_d2, 'CI_high': ci_high_d2, 'p': p_d2})

# --- Resumo tabular ---
print("\n" + "="*70)
print("RESUMO DE ROBUSTEZ")
print("="*70)
robustness_df = pd.DataFrame(robustness_results)
if len(robustness_df) > 0:
    robustness_df['Significant'] = robustness_df['p'].apply(lambda p: 'Yes' if p < 0.05 else ('No' if not np.isnan(p) else 'N/A'))
    print(robustness_df[['Check', 'OR', 'CI_low', 'CI_high', 'p', 'Significant']].round(3))
    print("\nInterpretação: Se OR é qualitativamente similar (direção, significância) em todas as verificações,")
    print("               os resultados são robustos a escolhas analíticas.")

In [ ]:
# === Forest plot dos resultados de robustez ===
if len(robustness_df) > 0:
    # Remover linhas sem CI (como Probit)
    rob_plot = robustness_df[robustness_df['CI_low'].notna()].copy()
    
    if len(rob_plot) > 0:
        fig, ax = plt.subplots(figsize=(10, max(6, len(rob_plot) * 0.4)))
        
        # Ordenar por OR
        rob_plot = rob_plot.sort_values('OR', ascending=True).reset_index(drop=True)
        
        y_pos = np.arange(len(rob_plot))
        ors = rob_plot['OR'].values
        ci_lows = rob_plot['CI_low'].values
        ci_highs = rob_plot['CI_high'].values
        labels = rob_plot['Check'].values
        
        # Erros para errorbar
        errors = np.array([[ors[i] - ci_lows[i], ci_highs[i] - ors[i]] for i in range(len(ors))]).T
        
        # Cores: verde se sig, cinza se não-sig, vermelho se placebo sig (problema)
        colors = []
        for idx, row in rob_plot.iterrows():
            if 'Placebo' in row['Check']:
                colors.append('red' if row['p'] < 0.05 else 'gray')
            else:
                colors.append('green' if row['p'] < 0.05 else 'lightgray')
        
        ax.errorbar(ors, y_pos, xerr=errors, fmt='o', markersize=8, capsize=5, capthick=2, 
                    ecolor=colors, markerfacecolor=colors, markeredgecolor='black')
        
        # Linha de referência OR=1
        ax.axvline(x=1.0, color='black', linestyle='--', linewidth=1, label='OR = 1 (no effect)')
        
        ax.set_yticks(y_pos)
        ax.set_yticklabels(labels)
        ax.set_xlabel('Odds Ratio (OR)')
        ax.set_title('Forest Plot: Robustness Checks for AI Exposure → exec_ok')
        ax.legend()
        ax.grid(axis='x', alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print("\nLegenda:")
        print("  Verde: p < 0.05 (significativo)")
        print("  Cinza: p >= 0.05 (não significativo)")
        print("  Vermelho: Placebo significativo (PROBLEMA - não esperado!)")
    else:
        print("No robustness results with CIs available for forest plot")
else:
    print("No robustness results to plot")


## 8. Interpretação e Discussão dos Resultados

### 8.1 Síntese dos achados principais

**RQ4**: *Em que medida o uso de ferramentas de IA na geração de código influencia a qualidade e a reprodutibilidade dos notebooks?*

Com base nas análises realizadas, reportamos os efeitos estimados de exposição à IA (medida por proxies heurísticos) sobre:

1. **Execução bem-sucedida** (`exec_ok`): notebooks com sinais de IA têm chances **[interpretar OR do modelo]** de executar sem erros em ambiente headless, controlando para popularidade do repositório, tamanho, dependências e completude prévia.

2. **Reprodutibilidade de outputs** (`outputs_equal`): quando o notebook original está disponível, notebooks com sinais de IA têm chances **[interpretar OR do modelo]** de reproduzir exatamente os mesmos outputs.

### 8.2 Conversão de OR para probabilidades (interpretação prática)

**Exemplo de interpretação** (ajustar com valores reais do modelo):

Suponha OR(ai_exposure → exec_ok) = 1.25 (IC95% [1.10, 1.42], p < 0.01):

- **Interpretação multiplicativa**: Notebooks com sinais de IA têm 25% mais chances (*odds*) de executar com sucesso.

- **Interpretação em probabilidades**: 
  - Se a probabilidade baseline (sem IA) de `exec_ok` é 50%, então:
    - Baseline odds = 0.50 / 0.50 = 1.0
    - Com IA: odds = 1.0 × 1.25 = 1.25
    - Prob com IA = 1.25 / (1 + 1.25) = 0.556 → **~56%**
  - **Diferença absoluta**: +6 pontos percentuais

- **Significância prática**: Pequeno efeito absoluto, mas pode ser relevante em populações grandes (e.g., milhares de notebooks).

### 8.3 Comparação com literatura

**Achados consistentes?**
- Pimentel et al. (2019) reportam ~40% de notebooks com execução completa em GitHub; nossos dados [comparar].
- Wang et al. (2024) identificam padrões estilísticos em código gerado por IA; nossas heurísticas capturam apenas sinais superficiais.

**Explicações possíveis para efeitos observados:**

**Se efeito positivo (OR > 1):**
- IA gera código mais estruturado, com menos erros sintáticos
- Usuários de IA documentam melhor (comentários, markdown)
- Viés de seleção: usuários sofisticados usam tanto IA quanto melhores práticas

**Se efeito negativo (OR < 1):**
- Código gerado por IA pode ter dependências hardcoded ou não-portáveis
- IA pode gerar código que "parece" correto mas falha em edge cases
- Over-reliance em IA reduz entendimento profundo do usuário

**Se efeito nulo (OR ≈ 1, p > 0.05):**
- Proxies heurísticos muito imprecisos (alta contaminação/perda)
- IA afeta qualidade apenas em dimensões não capturadas
- Efeitos positivos e negativos se cancelam

### 8.4 Significância estatística vs. prática

**Critérios de avaliação:**
- **Estatística**: p < 0.05, IC95% não cruza 1.0 → rejeita H0: OR = 1
- **Prática**: |OR - 1| > threshold (e.g., OR < 0.8 ou OR > 1.25 para efeito "relevante")

**Decisão**: [Preencher após ver resultados]
- Se estatisticamente significativo mas OR próximo de 1.0: efeito detectável mas pequeno
- Se não-significativo: ausência de evidência ≠ evidência de ausência (poder estatístico?)

### 8.5 Robustez dos achados

**Verificações realizadas** (Seção 7):
- Especificações alternativas (sem mediador, modelos probit)
- Definições alternativas de exposição (threshold, apenas marcadores diretos)
- Composições amostrais (balanceado, estratificado, sem outliers)
- Inferência (bootstrap, placebo test)

**Conclusão de robustez**: [Preencher após ver forest plot]
- Se OR mantém direção e significância na maioria das verificações → **robusto**
- Se apenas algumas verificações são significativas → **frágil, interpretar com cautela**
- Se placebo test é significativo → **PROBLEMA: sinal espúrio, não confiar nos resultados**

## 9. Ameaças à Validade

Seguindo Wohlin et al. (2012), classificamos ameaças em quatro categorias:

### 9.1 Validade Interna
**Confundimento residual**: Variáveis não observadas (habilidade, propósito, domínio) podem estar correlacionadas com IA e desfechos.

**Causalidade reversa**: Design cross-sectional não permite inferência causal.

### 9.2 Validade Externa
**Amostra não-representativa**: Apenas GitHub público; não generaliza para contextos privados/corporativos.

**Recorte temporal**: Ferramentas de IA evoluem rapidamente; resultados podem desatualizar.

### 9.3 Validade de Construto
**Mensuração imperfeita de IA**: Proxies heurísticos têm falsos positivos/negativos desconhecidos.

**Quality score**: Pesos baseados em julgamento teórico; componentes reportados separadamente para transparência.

### 9.4 Validade de Conclusão
**Poder estatístico**: Depende da prevalência de IA; effect sizes reportados.

**Violação de pressupostos**: VIF, log-transforms e modelos alternativos testados.


## 10. Referências Bibliográficas

### Metodologia de pesquisa empírica
- **Wohlin, C., Runeson, P., Höst, M., Ohlsson, M. C., Regnell, B., & Wesslén, A.** (2012). *Experimentation in software engineering*. Springer Science & Business Media.
- **Shadish, W. R., Cook, T. D., & Campbell, D. T.** (2002). *Experimental and quasi-experimental designs for generalized causal inference*. Houghton Mifflin.
- **Creswell, J. W., & Creswell, J. D.** (2017). *Research design: Qualitative, quantitative, and mixed methods approaches*. Sage publications.

### Qualidade e reprodutibilidade de notebooks
- **Pimentel, J. F., Murta, L., Braganholo, V., & Freire, J.** (2019). A large-scale study about quality and reproducibility of Jupyter notebooks. In *2019 IEEE/ACM 16th International Conference on Mining Software Repositories (MSR)* (pp. 507-517). IEEE.
- **Rule, A., Birmingham, A., Zuniga, C., Altintas, I., Huang, S. C., Knight, R., ... & Crouch, S. R.** (2019). Ten simple rules for writing and sharing computational analyses in Jupyter Notebooks. *PLoS computational biology*, 15(7), e1007007.
- **Pimentel, J. F., Murta, L., Braganholo, V., & Freire, J.** (2021). Understanding and improving the quality and reproducibility of Jupyter notebooks. *Empirical Software Engineering*, 26(4), 1-55.

### Detecção de código gerado por IA
- **Wang, Y., et al.** (2024). An empirical study on automatically detecting AI-generated source code: How far are we? *arXiv preprint arXiv:2411.04299*.

### Boas práticas computacionais
- **Wilson, G., Aruliah, D. A., Brown, C. T., Hong, N. P. C., Davis, M., Guy, R. T., ... & Wilson, P.** (2014). Best practices for scientific computing. *PLoS biology*, 12(1), e1001745.
- **Sandve, G. K., Nekrutenko, A., Taylor, J., & Hovig, E.** (2013). Ten simple rules for reproducible computational research. *PLoS computational biology*, 9(10), e1003285.

### Effect sizes e meta-análise
- **Cohen, J.** (1988). *Statistical power analysis for the behavioral sciences* (2nd ed.). Lawrence Erlbaum Associates.


## 11. Reprodutibilidade e Disponibilidade de Dados

### 11.1 Computational environment

**Software dependencies**:
```
Python 3.8+
pandas >= 1.3.0
numpy >= 1.20.0
matplotlib >= 3.3.0
scipy >= 1.7.0
statsmodels >= 0.13.0
scikit-learn >= 0.24.0
nbformat >= 5.0
nbclient >= 0.5
```

Instale com: `pip install -r requirements.txt`

**Hardware**: Análises executadas em ambiente padrão (CPU, 8GB RAM); nenhuma GPU necessária.

### 11.2 Random seeds

Todas as operações estocásticas usam seeds fixas para reprodutibilidade:
- Bootstrap resampling: `random_state=42+i` (loop indexado)
- Permutation tests: `np.random.default_rng(42)`
- Sample balancing: `random_state=123`

### 11.3 Data availability

**Input data**:
- `collection.csv`: Metadados de notebooks coletados via `collect_notebooks.py`
- `execution_results.csv`: Resultados de execução via `execute_notebooks.py`

### 11.4 Code availability

**Repositório**: https://github.com/andre-fig/reproducibility_of_jupyter_notebooks


### 11.5 Replication package

Para replicar as análises:
1. Clone o repositório: `git clone https://github.com/andre-fig/reproducibility_of_jupyter_notebooks`
2. Instale dependências: `pip install -r requirements.txt`
3. Baixe os dados brutos ou execute coleta: `python scripts/collect_notebooks.py --date-start ... --date-end ... --output data/outputs/collection.csv`
4. Execute notebooks: `python scripts/execute_notebooks.py --input data/outputs/collection.csv --output data/outputs/execution_results.csv`
5. Execute este notebook: `jupyter notebook RQ4_IA_qualidade_reprodutibilidade.ipynb`



## Apêndice: Dicionário de Variáveis

### A.1 Exposição à IA (Seção 2)
| Variável | Tipo | Descrição | Fonte |
|----------|------|-----------|-------|
| `ai_marker_found` | Binary (0/1) | Menção explícita de IA em markdown (ChatGPT, Copilot, etc.) | Coleta (heurística) |
| `triple_backticks_in_code` | Binary (0/1) | Artefatos de cópia-cola (```python em células de código) | Coleta (heurística) |
| `ai_comment_density_norm` | Continuous (0-1) | Proporção markdown/código normalizada | Derivada |
| `ai_style_uniformity` | Continuous (0-1) | Ordem de execução limpa (proxy de uniformidade) | Derivada |
| `ai_pattern_score` | Continuous (0-1) | Escore composto ponderado de sinais de IA | Derivada |
| `ai_exposure_binary` | Binary (0/1) | Qualquer sinal direto de IA (marcador OU backticks) | Derivada |
| `ai_exposure_level` | Ordinal (0-3) | 0=none, 1=weak, 2=moderate, 3=strong | Derivada |

### A.2 Qualidade de Notebooks (Seção 3)
| Variável | Tipo | Descrição | Peso no composite |
|----------|------|-----------|-------------------|
| `quality_d1_execution` | Continuous (0-1) | % de código executado | 30% |
| `quality_d2_structure` | Binary (0/1) | Ordem de execução sem ambiguidades | 20% |
| `quality_d3_no_errors` | Binary (0/1) | Ausência de outputs de erro | 20% |
| `quality_d4_testing` | Binary (0/1) | Importa módulos de teste (pytest, unittest) | 15% |
| `quality_d5_deps` | Binary (0/1) | Presença de requirements.txt/setup.py/Pipfile | 10% |
| `quality_d6_portable` | Binary (0/1) | Ausência de caminhos absolutos | 5% |
| `quality_penalty_skips` | Continuous (0-1) | Penalização por gaps na execução | 10% |
| `quality_score` | Continuous (0-1) | Composite score ponderado | 100% |

### A.3 Reprodutibilidade (Seção 4)
| Variável | Tipo | Descrição | Fonte |
|----------|------|-----------|-------|
| `exec_ok` | Binary (0/1) | Execução completa sem erro (headless, timeout 300s) | Executor |
| `original_found` | Binary (0/1) | Notebook original salvo disponível para comparação | Executor |
| `outputs_equal` | Binary (0/1) | Hash de outputs original == executado (condicional a `original_found`) | Executor |
| `outputs_hash_orig` | String (SHA256) | Hash dos outputs canonicalizados do notebook original | Executor |
| `outputs_hash_exec` | String (SHA256) | Hash dos outputs canonicalizados após execução | Executor |

### A.4 Covariáveis de Controle
| Variável | Tipo | Descrição | Transformação |
|----------|------|-----------|---------------|
| `repo_full_name` | String | owner/repo no GitHub | - |
| `repo_stars` | Integer | Número de stars do repositório | Log-transform: `log(stars + 1)` |
| `repo_stars_log` | Continuous | Log-stars (para regressão) | Derivada |
| `n_code` | Integer | Número de células de código no notebook | Log-transform: `log(n_code)` |
| `n_code_log` | Continuous | Log do número de células (para regressão) | Derivada |
| `deps_any` | Binary (0/1) | Presença de qualquer arquivo de dependências | Coleta |
| `percent_code_executed` | Continuous (0-100) | % de células com execution_count definido | Coleta |

### A.5 Variáveis Auxiliares
| Variável | Tipo | Descrição |
|----------|------|-----------|
| `n_markdown` | Integer | Número de células markdown |
| `has_unambiguous_order` | Binary (0/1) | Execução sequencial sem gaps ou repetições |
| `out_of_order` | Binary (0/1) | Células executadas fora de ordem |
| `n_skips_total` | Integer | Número total de gaps na sequência de execution_count |
| `outputs_error` | Binary (0/1) | Pelo menos um output do tipo "error" |
| `uses_testing_module` | Binary (0/1) | Importa pytest, unittest, nose, etc. |
| `has_abs_data_path` | Binary (0/1) | Código contém caminhos absolutos (/path ou C:\path) |

### A.6 Notas metodológicas
- **Missing data**: NAs tratados como 0 para binárias, imputados ou excluídos para contínuas
- **Outliers**: Winsorização aplicada onde apropriado (e.g., stars e n_code log-transformed)
- **Normalização**: Variáveis contínuas escaladas para [0,1] quando usadas em composites


## Apêndice: dicionário de variáveis (principal)
- `ai_exposure` (0/1): heurística (marcadores de IA ou cercas ```python coladas);
- `quality_score` (0–1): escore composto (ajuste os pesos conforme sua rubrica);
- `exec_ok` (0/1): execução bem-sucedida *headless*;
- `outputs_equal` (0/1): reprodutibilidade por igualdade de hash de outputs quando há original salvo;
- `repo_stars`, `n_code`, `deps_any`, `percent_code_executed`: *covariates* de controle.